# Project D — Task **(b)**: Most specific / leaf-level classification

**Assignment:** assign articles to the **most specific** topic levels possible (full hierarchy, leaf-level evaluation).

This notebook loads the same pool and tree, then:
- **Leaf-level metrics** (test): multilabel leaf F1, path-to-gold branching recall — for **all three** linear baselines with **H1-only** training (same as the exploratory leaf section in the combined demo).
- **Full tree:** train **every** branching edge in the taxonomy (`binary_edge_specs`), with **verbose progress** per edge, then overall **`ft_*`** metrics and leaf scores on the **test** split — again for all three models.

For **task (a)** (four top-level topics only), see **`hierarchical_demo_1a.ipynb`**.

**Working directory:** `Homework 4`. **Warning:** full-tree training is **slow** (many edges × 3 models).

In [1]:
from pathlib import Path

from topic_hierarchy import binary_edge_specs, load_topic_tree, summary


def homework4_base() -> Path:
    """Resolve Homework 4 folder (contains topics.csv)."""
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — set cwd to Homework 4 or edit BASE")


BASE = homework4_base()
BASE

PosixPath('/Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4')

## Topic tree summary

In [2]:
tree = load_topic_tree(str(BASE / "topics.csv"))
s = summary(tree)
for k, v in sorted(s.items()):
    print(f"  {k}: {v}")

  max_branching_factor: 23
  max_depth_from_root: 5
  n_binary_edge_classifiers: 103
  n_branching_nodes: 22
  n_leaves: 82
  n_local_classifiers: 22
  n_nodes: 117
  traversal_root: Root


## Binary edge inventory (sample)

Each row is one trainable **binary** model: membership in the child’s subtree. Total count is **`n_binary_edge_classifiers`** in the summary above.

In [3]:
edges = binary_edge_specs(tree)
print(f"Total binary edges: {len(edges)}\n")
for sp in edges[:10]:
    print(f"  {sp.parent!r} -> {sp.child!r}  (depth {sp.depth})")

Total binary edges: 103

  'Root' -> 'CCAT'  (depth 0)
  'Root' -> 'ECAT'  (depth 0)
  'Root' -> 'GCAT'  (depth 0)
  'Root' -> 'MCAT'  (depth 0)
  'CCAT' -> 'C1'  (depth 1)
  'CCAT' -> 'C2'  (depth 1)
  'CCAT' -> 'C3'  (depth 1)
  'CCAT' -> 'C4'  (depth 1)
  'ECAT' -> 'E1'  (depth 1)
  'ECAT' -> 'E2'  (depth 1)


In [ ]:
from hierarchical_training_data import make_multilabel_binary_pool_data

mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

## Leaf-level evaluation (test split)

**How this relates to “cutting” early**

- **Per-edge training labels** already encode subtree membership: for edge `parent → child`, gold is 1 iff the article has *any* topic in `child`’s subtree. Separate per-edge precision/recall (elsewhere) score each binary head on its own.
- **Inference** (`predict_reached_nodes`): at a **branching** parent, you only go down `parent → child` if that edge’s model predicts **1**. If it predicts **0**, you **do not** open that branch — the walk stops exploring that side (your “cut”). Unary edges (only child) are followed automatically — no binary there.
- **Leaf multilabel F1** compares gold **leaf** codes (`news_topics` ∩ taxonomy leaves) to **`predict_reached_leaves`**. If you cut too high, predicted leaves miss gold leaves → **recall** drops; spurious deep positives hurt **precision**.
- **Path-to-gold branching recall** (`path_gold_branch_recall`): for each article, take all **gold leaf** topics; for the union of **branching** edges on paths `Root → … → leaf`, check whether each required edge predicts **1**. A **0** or a **missing** fitted model on such an edge is counted as wrong — that is exactly “should have progressed through this node for this gold leaf but did not.”

**Per-edge FNs are only on edges that should fire:** for each `parent → child`, gold is **1** iff the article has a label in that child’s subtree (`gold_positive_edge`). Predicting **0** on an edge with gold **0** is a true negative, not an FN. So if “none of the children” fire but only **one** subtree was actually required, only that edge’s head gets an FN — not one FN per sibling.

Together: leaf F1 summarizes **leaf set** agreement; path recall scores **routing** on paths to gold leaves; standard per-edge metrics (elsewhere) already handle **should-have-gone-forward** FNs. With only H1 (or partial trees) fitted, path recall and leaf scores will be **pessimistic** until deeper edges exist.

In [ ]:
import time

from hierarchical_summary_metrics import comparison_table, linear_model_factories, train_and_summarize

# Same three linear baselines; adds leaf multilabel F1 + path-to-gold branching recall (test only).
# Slower than task 1a because it scores leaves + paths per model.
factories = linear_model_factories(max_features=8000)
n_models = len(factories)
print(
    f"Leaf + path metrics (test): {n_models} models, fit H1 on train, eval on test.\n",
    flush=True,
)

leaf_rows = []
for i, (model_name, factory) in enumerate(factories.items(), start=1):
    print(f"[{i}/{n_models}] {model_name} ...", flush=True)
    t0 = time.perf_counter()
    _, row = train_and_summarize(
        model_name, tree, mldata, factory, split="test", include_leaf=True
    )
    print(f"    ok ({time.perf_counter() - t0:.1f}s)", flush=True)
    leaf_rows.append(row)

leaf_df = comparison_table(leaf_rows)
num_cols = [c for c in leaf_df.columns if c != "model"]
out_leaf = leaf_df.copy()
out_leaf[num_cols] = out_leaf[num_cols].round(4)
out_leaf

Leaf + path metrics (test): 3 models, fit H1 on train, eval on test.

[1/3] LinearSVC ...
    ok (21.6s)
[2/3] GLMNet_elasticnet ...
    ok (37.4s)
[3/3] MaxEnt_L2 ...
    ok (21.8s)


,model,h1_macro_f1,h1_pooled_micro_f1,h1_pos_weighted_f1,leaf_leaf_micro_f1,leaf_leaf_macro_f1,leaf_leaf_weighted_f1,leaf_leaf_samples_f1
0,LinearSVC,0.8671,0.8802,0.8798,0.0,0.0,0.0,0.0
1,GLMNet_elasticnet,0.8575,0.8786,0.8770,0.0,0.0,0.0,0.0
2,MaxEnt_L2,0.8608,0.8814,0.8798,0.0,0.0,0.0,0.0


## Full tree: train all binary edges + overall / leaf metrics (test)

**Training:** every branching edge in `topics.csv` (`binary_edge_specs`), one model per `parent → child`, same pool and gold as elsewhere. Skips an edge if the **train** split has only one class. Progress is printed as `[i/N] parent → child (depth d)` so you can see where a long run is.

**Overall edge metrics (test, `ft_*`):**

- **`ft_pooled_micro_f1`**: one F1 over **all** (document × edge) pairs — micro-averaged over every binary decision.
- **`ft_macro_f1`**: unweighted mean of per-edge F1 (each edge counts equally).
- **`ft_pos_weighted_f1`**: mean of per-edge F1 weighted by the number of **positive** gold labels on that edge in the test split (edges with more positives get more weight).

**Leaf:** same `leaf_*` and `path_*` metrics as the earlier leaf section, now with a **fully trained** router so routing can reach deep leaves.

**Models:** same three linear factories (`LinearSVC`, `GLMNet_elasticnet`, `MaxEnt_L2`). **This is very slow** (many edges × TF-IDF fits × 3 models); run when ready.

In [ ]:
import time

from hierarchical_summary_metrics import (
    comparison_table,
    linear_model_factories,
    train_full_tree_and_summarize,
)

# Full taxonomy: all binary edges, all three models, test metrics + leaf/path.
factories = linear_model_factories(max_features=8000)
full_tree_rows = []
for mi, (model_name, factory) in enumerate(factories.items(), start=1):
    print("\n" + "=" * 72, flush=True)
    print(f"MODEL [{mi}/{len(factories)}] {model_name}", flush=True)
    print("=" * 72 + "\n", flush=True)
    t_wall = time.perf_counter()
    _, row = train_full_tree_and_summarize(
        model_name,
        tree,
        mldata,
        factory,
        split="test",
        verbose=True,
    )
    print(
        f"\n>>> {model_name} finished in {time.perf_counter() - t_wall:.1f}s wall time\n",
        flush=True,
    )
    full_tree_rows.append(row)

full_tree_df = comparison_table(full_tree_rows)
_num = [c for c in full_tree_df.columns if c != "model"]
_ft = full_tree_df.copy()
_ft[_num] = _ft[_num].round(4)
_ft

ImportError: cannot import name 'train_full_tree_and_summarize' from 'hierarchical_summary_metrics' (/Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/hierarchical_summary_metrics.py)